In [37]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [38]:
df=pd.read_csv("car_prediction_data.csv")

In [39]:
df.head()

,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


In [40]:
df = df.dropna()
current_year = 2026
df['age'] = current_year - df['Year']
df = df.drop('Year', axis=1)

print("Data Prepared:")
print(df.head())

Data Prepared:
  Car_Name  Selling_Price  Present_Price  Kms_Driven Fuel_Type Seller_Type  \
0     ritz           3.35           5.59       27000    Petrol      Dealer   
1      sx4           4.75           9.54       43000    Diesel      Dealer   
2     ciaz           7.25           9.85        6900    Petrol      Dealer   
3  wagon r           2.85           4.15        5200    Petrol      Dealer   
4    swift           4.60           6.87       42450    Diesel      Dealer   

  Transmission  Owner  age  
0       Manual      0   12  
1       Manual      0   13  
2       Manual      0    9  
3       Manual      0   15  
4       Manual      0   12  


In [41]:
df.columns = df.columns.str.strip()

print("Actual columns in DataFrame:", df.columns.tolist())

Actual columns in DataFrame: ['Car_Name', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'age']


In [42]:
X = df.drop('Selling_Price', axis=1)
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [43]:
numeric_features = ['Present_Price', 'Kms_Driven', 'Owner', 'age']
categorical_features = ['Fuel_Type', 'Seller_Type', 'Transmission']

In [44]:

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])


In [45]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42))
])

In [46]:
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)

# THIS SHOULD NOW WORK
grid_search.fit(X_train, y_train)
print("Grid search complete!")

Grid search complete!


In [48]:
# 6. HYPERPARAMETER TUNING
param_grid = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
print(f"Best Parameters found: {grid_search.best_params_}")

Best Parameters found: {'regressor__max_depth': None, 'regressor__min_samples_split': 2, 'regressor__n_estimators': 100}


In [ ]:
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("--- EVALUATION REPORT ---")
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Root Mean Squared Error: {rmse:.4f}")
print(f"R2 Score (Accuracy): {r2:.4f}")

joblib.dump(best_model, 'car_price_model.pkl')
print("\nSuccess: Model saved as 'car_price_model.pkl'")

--- EVALUATION REPORT ---
Mean Absolute Error: 0.6094
Root Mean Squared Error: 0.9369
R2 Score (Accuracy): 0.9619

Success: Model saved as 'car_price_model.pkl'
